In [1]:
# Task 6: Real-Time ML Inference REST API

import os
import sys

print("Task 6 notebook is ready!")
print("Python version:", sys.version.split()[0])
print("Working directory:", os.getcwd())

Task 6 notebook is ready!
Python version: 3.13.16
Working directory: /content


In [2]:
# Task 6: Check required libraries

import tensorflow as tf
import fastapi
import pydantic
import pytest

print("TensorFlow version:", tf.__version__)
print("FastAPI version:", fastapi.__version__)
print("Pydantic version:", pydantic.__version__)
print("Pytest version:", pytest.__version__)

print("\nAll required libraries are available!")

TensorFlow version: 2.21.0
FastAPI version: 0.141.1
Pydantic version: 2.13.5
Pytest version: 8.4.2

All required libraries are available!


In [3]:
import os
import tensorflow as tf

# Download and extract the IMDB movie review dataset
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

dataset_path = tf.keras.utils.get_file(
    fname="aclImdb_v1.tar.gz",
    origin=url,
    untar=True,
    cache_dir="/content",
    cache_subdir=""
)

# Locate the extracted dataset
dataset_dir = os.path.join("/content", "aclImdb")

print("Dataset download completed!")
print("Dataset directory:", dataset_dir)
print("Directory exists:", os.path.exists(dataset_dir))

if os.path.exists(dataset_dir):
    print("Available folders:", os.listdir(dataset_dir))

84125825/84125825 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Dataset download completed!
Dataset directory: /content/aclImdb
Directory exists: False


In [4]:
import os

# Search common Colab locations for the IMDB training folder
for base in ["/content", "/root/.keras/datasets"]:
    print(f"\nChecking: {base}")

    if os.path.exists(base):
        for root, dirs, files in os.walk(base):
            if os.path.basename(root) == "train":
                if "pos" in dirs and "neg" in dirs:
                    print("Possible IMDB training folder:", root)


Checking: /content
Possible IMDB training folder: /content/aclImdb_v1_extracted/aclImdb/train

Checking: /root/.keras/datasets


In [5]:
import os
import pandas as pd

# Correct dataset location
dataset_dir = "/content/aclImdb_v1_extracted/aclImdb"

def load_reviews(folder):
    reviews = []
    labels = []

    for sentiment, label in [("pos", 1), ("neg", 0)]:
        sentiment_dir = os.path.join(folder, sentiment)

        for filename in os.listdir(sentiment_dir):
            if filename.endswith(".txt"):
                filepath = os.path.join(sentiment_dir, filename)

                with open(filepath, "r", encoding="utf-8") as f:
                    reviews.append(f.read())

                labels.append(label)

    return pd.DataFrame({
        "text": reviews,
        "label": labels
    })

# Load training and testing data
train_df = load_reviews(os.path.join(dataset_dir, "train"))
test_df = load_reviews(os.path.join(dataset_dir, "test"))

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

print("\nSample reviews:")
print(train_df.head(3))

print("\nLabel counts:")
print(train_df["label"].value_counts())

Training samples: 25000
Testing samples: 25000

Sample reviews:
                                                text  label
0  This is a weird movie about an archaeologist s...      1
1  This movie is basically a documentary of the c...      1
2  I guess I'm part of the silent minority who en...      1

Label counts:
label
1    12500
0    12500
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

# Separate review text and sentiment labels
x_train_all = train_df["text"].astype(str).values
y_train_all = train_df["label"].values

x_test = test_df["text"].astype(str).values
y_test = test_df["label"].values

# Split training data into training and validation sets
x_train, x_val, y_train, y_val = train_test_split(
    x_train_all,
    y_train_all,
    test_size=0.2,
    random_state=42,
    stratify=y_train_all
)

print("Training samples:", len(x_train))
print("Validation samples:", len(x_val))
print("Testing samples:", len(x_test))
print("\nData preparation completed!")

Training samples: 20000
Validation samples: 5000
Testing samples: 25000

Data preparation completed!


In [7]:
import tensorflow as tf

# Configure text preprocessing
max_tokens = 20000
sequence_length = 200

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Learn the vocabulary using training reviews only
vectorizer.adapt(x_train)

# Convert reviews into numerical sequences
x_train_vec = vectorizer(x_train)
x_val_vec = vectorizer(x_val)
x_test_vec = vectorizer(x_test)

print("Vocabulary size:", len(vectorizer.get_vocabulary()))
print("Training data shape:", x_train_vec.shape)
print("Validation data shape:", x_val_vec.shape)
print("Testing data shape:", x_test_vec.shape)
print("\nText vectorization completed!")

Vocabulary size: 20000
Training data shape: (20000, 200)
Validation data shape: (5000, 200)
Testing data shape: (25000, 200)

Text vectorization completed!


In [8]:
import tensorflow as tf

# Build the sentiment classification model
classifier = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(sequence_length,)),
    tf.keras.layers.Embedding(
        input_dim=max_tokens,
        output_dim=64
    ),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# Configure the model for training
classifier.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Display the model architecture
classifier.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,284,225 (4.90 MB)

 Trainable params: 1,284,225 (4.90 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Train the sentiment classifier

history = classifier.fit(
    x_train_vec,
    y_train,
    validation_data=(x_val_vec, y_val),
    epochs=5,
    batch_size=64,
    verbose=1
)

print("\nModel training completed!")

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.7398 - loss: 0.5307 - val_accuracy: 0.8432 - val_loss: 0.3663
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.8771 - loss: 0.2961 - val_accuracy: 0.8502 - val_loss: 0.3353
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9072 - loss: 0.2349 - val_accuracy: 0.8602 - val_loss: 0.3236
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.9398 - loss: 0.1693 - val_accuracy: 0.8786 - val_loss: 0.3086
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.9484 - loss: 0.1423 - val_accuracy: 0.8738 - val_loss: 0.3229

Model training completed!


In [10]:
# Evaluate the trained classifier on unseen test reviews

test_loss, test_accuracy = classifier.evaluate(
    x_test_vec,
    y_test,
    verbose=1
)

print("\n========== MODEL EVALUATION ==========")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print("======================================")

782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8456 - loss: 0.3928

========== MODEL EVALUATION ==========
Test Loss: 0.3928
Test Accuracy: 0.8456
Test Accuracy: 84.56%


In [11]:
import tensorflow as tf
import os

# Create an input that accepts raw text
text_input = tf.keras.Input(
    shape=(),
    dtype=tf.string,
    name="text"
)

# Connect text preprocessing to the trained classifier
vectorized_text = vectorizer(text_input)
sentiment_probability = classifier(vectorized_text)

# Create one model for API inference
serving_model = tf.keras.Model(
    inputs=text_input,
    outputs=sentiment_probability,
    name="imdb_sentiment_api"
)

print("Combined model created successfully!")
print("Input:", serving_model.input)
print("Output:", serving_model.output)

Combined model created successfully!
Input: <KerasTensor shape=(None,), dtype=string, sparse=False, ragged=False, name=text>
Output: <KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor_6>


In [12]:
import os
import tensorflow as tf

# Create a folder for our model
os.makedirs("/content/models", exist_ok=True)

# Save the combined model, including text preprocessing
model_path = "/content/models/imdb_sentiment_api.keras"

serving_model.save(model_path)

# Verify that the file was saved
print("Model saved successfully:", os.path.exists(model_path))
print(
    "Model size:",
    round(os.path.getsize(model_path) / (1024 * 1024), 2),
    "MB"
)
print("Model path:", model_path)

Model saved successfully: True
Model size: 15.04 MB
Model path: /content/models/imdb_sentiment_api.keras


In [14]:
import tensorflow as tf
import numpy as np

# Load the saved model
loaded_model = tf.keras.models.load_model(
    "/content/models/imdb_sentiment_api.keras"
)

# Sample movie reviews
reviews = [
    "This movie was wonderful, entertaining, and beautifully acted!",
    "This was a terrible movie. The story was boring and disappointing."
]

# Explicitly convert the reviews to TensorFlow strings
review_tensor = tf.constant(reviews, dtype=tf.string)

# Predict sentiment
probabilities = loaded_model.predict(
    review_tensor,
    verbose=0
).flatten()

# Display predictions
for review, probability in zip(reviews, probabilities):
    sentiment = "Positive" if probability >= 0.5 else "Negative"

    print("Review:", review)
    print("Sentiment:", sentiment)
    print(f"Positive probability: {float(probability):.4f}")
    print("-" * 60)

print("Saved model loaded and tested successfully!")

Review: This movie was wonderful, entertaining, and beautifully acted!
Sentiment: Positive
Positive probability: 0.8527
------------------------------------------------------------
Review: This was a terrible movie. The story was boring and disappointing.
Sentiment: Negative
Positive probability: 0.0286
------------------------------------------------------------
Saved model loaded and tested successfully!


In [15]:
%pip install fastapi uvicorn

In [16]:
%%writefile /content/main.py

import os
import numpy as np
import tensorflow as tf

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

# Saved model location
MODEL_PATH = "/content/models/imdb_sentiment_api.keras"

app = FastAPI(
    title="IMDB Sentiment Analysis API",
    description="Predicts the sentiment of movie reviews.",
    version="1.0.0"
)

# Load the model when the application starts
model = None


@app.on_event("startup")
def load_model():
    global model

    if not os.path.exists(MODEL_PATH):
        raise RuntimeError(f"Model file not found: {MODEL_PATH}")

    model = tf.keras.models.load_model(MODEL_PATH)


# Request schema
class PredictRequest(BaseModel):
    text: str = Field(min_length=1, max_length=10000)


# Response schema
class PredictResponse(BaseModel):
    prediction: str
    positive_probability: float
    negative_probability: float


@app.get("/")
def home():
    return {"message": "IMDB Sentiment Analysis API is running"}


@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": model is not None}


@app.post("/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    if model is None:
        raise HTTPException(
            status_code=503,
            detail="Model is not loaded"
        )

    text_tensor = tf.constant([request.text], dtype=tf.string)
    score = float(model.predict(text_tensor, verbose=0).reshape(-1)[0])

    prediction = "positive" if score >= 0.5 else "negative"

    return {
        "prediction": prediction,
        "positive_probability": score,
        "negative_probability": 1.0 - score
    }

Writing /content/main.py


In [17]:
import os

print("Checking Task 6 files...\n")

files_to_check = [
    "/content/main.py",
    "/content/imdb_sentiment_api.keras",
    "/content/requirements.txt"
]

for file_path in files_to_check:
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"✅ Found: {file_path} ({size_mb:.2f} MB)")
    else:
        print(f"⚠️ Not found yet: {file_path}")

Checking Task 6 files...

✅ Found: /content/main.py (0.00 MB)
⚠️ Not found yet: /content/imdb_sentiment_api.keras
⚠️ Not found yet: /content/requirements.txt


In [18]:
# Save the combined model for the Task 6 API

model_path = "/content/imdb_sentiment_api.keras"

serving_model.save(model_path)

print("✅ Combined model saved successfully!")
print("Model path:", model_path)

import os
print("Model size:", round(os.path.getsize(model_path) / (1024 * 1024), 2), "MB")

✅ Combined model saved successfully!
Model path: /content/imdb_sentiment_api.keras
Model size: 15.04 MB


In [19]:
import tensorflow as tf
import os

model_path = "/content/imdb_sentiment_api.keras"

loaded_model = tf.keras.models.load_model(model_path)

reviews = tf.constant([
    "This movie was wonderful and entertaining!",
    "This movie was terrible and boring."
], dtype=tf.string)

probabilities = loaded_model(reviews, training=False).numpy().reshape(-1)

for review, probability in zip(reviews.numpy(), probabilities):
    label = "Positive" if probability >= 0.5 else "Negative"
    print("Review:", review.decode("utf-8"))
    print("Prediction:", label)
    print("Positive probability:", round(float(probability), 4))
    print()

print("✅ Saved model loaded and tested successfully!")

Review: This movie was wonderful and entertaining!
Prediction: Positive
Positive probability: 0.7972

Review: This movie was terrible and boring.
Prediction: Negative
Positive probability: 0.0741

✅ Saved model loaded and tested successfully!


In [20]:
%%writefile /content/requirements.txt
fastapi==0.115.12
uvicorn==0.34.0
tensorflow==2.16.1
pydantic==2.10.6
pytest==8.3.5
httpx==0.28.1

Writing /content/requirements.txt


In [21]:
import os
import shutil

# Create the project folders
project_dir = "/content/task6_project"

folders = [
    "app",
    "models",
    "tests"
]

for folder in folders:
    os.makedirs(os.path.join(project_dir, folder), exist_ok=True)

# Copy the saved model into the models folder
shutil.copy2(
    "/content/imdb_sentiment_api.keras",
    os.path.join(project_dir, "models", "imdb_sentiment_api.keras")
)

# Copy requirements.txt into the project folder
shutil.copy2(
    "/content/requirements.txt",
    os.path.join(project_dir, "requirements.txt")
)

print("✅ Project folders created!")
print("\nProject structure:")

for root, dirs, files in os.walk(project_dir):
    level = root.replace(project_dir, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}    {file}")

✅ Project folders created!

Project structure:
task6_project/
    requirements.txt
    app/
    tests/
    models/
        imdb_sentiment_api.keras


In [22]:
%%writefile /content/task6_project/app/main.py

import os
import tensorflow as tf

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

# Model location
MODEL_PATH = os.getenv(
    "MODEL_PATH",
    "models/imdb_sentiment_api.keras"
)

app = FastAPI(
    title="IMDB Sentiment Analysis API",
    description="Predicts whether a movie review is positive or negative.",
    version="1.0.0"
)

model = None


# Request format
class PredictRequest(BaseModel):
    text: str = Field(min_length=1, max_length=10000)


# Response format
class PredictResponse(BaseModel):
    prediction: str
    positive_probability: float
    negative_probability: float


# Load the trained model when the API starts
@app.on_event("startup")
def load_model():
    global model

    if not os.path.isabs(MODEL_PATH):
        base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
        model_path = os.path.join(base_dir, MODEL_PATH)
    else:
        model_path = MODEL_PATH

    if not os.path.exists(model_path):
        raise RuntimeError(f"Model file not found: {model_path}")

    model = tf.keras.models.load_model(model_path)


# Health-check endpoint
@app.get("/health")
def health():
    return {
        "status": "ok",
        "model_loaded": model is not None
    }


# Sentiment prediction endpoint
@app.post("/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    if model is None:
        raise HTTPException(
            status_code=503,
            detail="Model is not loaded"
        )

    text_tensor = tf.constant([request.text], dtype=tf.string)

    score = float(
        model(text_tensor, training=False).numpy().reshape(-1)[0]
    )

    prediction = "positive" if score >= 0.5 else "negative"

    return {
        "prediction": prediction,
        "positive_probability": score,
        "negative_probability": 1.0 - score
    }

Writing /content/task6_project/app/main.py


In [23]:
%%writefile /content/task6_project/tests/test_api.py

import sys
from pathlib import Path

import numpy as np
import pytest
from fastapi.testclient import TestClient

# Make the app module accessible to the tests
PROJECT_DIR = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_DIR))

import app.main as main


class FakeModel:
    """A lightweight model for testing API responses."""

    def __call__(self, text_tensor, training=False):
        return np.array([[0.8]], dtype=np.float32)


@pytest.fixture
def client(monkeypatch):
    # Use a fake model so tests don't need to load the real model.
    monkeypatch.setattr(main, "model", FakeModel())
    return TestClient(main.app)


def test_health_endpoint(client):
    response = client.get("/health")

    assert response.status_code == 200
    assert response.json() == {
        "status": "ok",
        "model_loaded": True
    }


def test_predict_response_schema(client):
    response = client.post(
        "/predict",
        json={"text": "This movie was fantastic!"}
    )

    assert response.status_code == 200

    data = response.json()
    assert data["prediction"] == "positive"
    assert "positive_probability" in data
    assert "negative_probability" in data


def test_probabilities_are_valid(client):
    response = client.post(
        "/predict",
        json={"text": "A wonderful movie."}
    )

    data = response.json()

    assert 0.0 <= data["positive_probability"] <= 1.0
    assert 0.0 <= data["negative_probability"] <= 1.0
    assert (
        data["positive_probability"] + data["negative_probability"]
    ) == pytest.approx(1.0, abs=1e-6)


def test_empty_text_is_rejected(client):
    response = client.post(
        "/predict",
        json={"text": ""}
    )

    assert response.status_code == 422

Writing /content/task6_project/tests/test_api.py


In [24]:
%%writefile /content/task6_project/Dockerfile

FROM python:3.11-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

# Install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy the application and trained model
COPY app ./app
COPY models ./models

# API port
EXPOSE 8000

# Start the FastAPI server
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing /content/task6_project/Dockerfile


In [25]:
%%writefile /content/task6_project/.dockerignore
.git
.gitignore
__pycache__/
*.py[cod]
.pytest_cache/
.ipynb_checkpoints/
.env
venv/
.venv/

Writing /content/task6_project/.dockerignore


In [26]:
%%writefile /content/task6_project/.gitignore
__pycache__/
*.py[cod]
.pytest_cache/
.env
venv/
.venv/
.ipynb_checkpoints/
.DS_Store

Writing /content/task6_project/.gitignore


In [27]:
%%writefile /content/task6_project/README.md

# Real-Time ML Inference REST API

## Project Overview

This project provides a REST API for classifying movie reviews as
positive or negative using a trained deep learning sentiment classifier.

The API is built with FastAPI and supports JSON requests and responses.

## Model Performance

- Dataset: IMDb movie reviews (ACL IMDb dataset)
- Task: Binary sentiment classification
- Test accuracy: 84.56%
- Output: Positive or negative sentiment with probabilities

## Architecture

1. A client sends a movie review as JSON.
2. FastAPI validates the incoming request.
3. The trained TensorFlow model processes the review.
4. The API returns the predicted sentiment and probabilities.

## Project Structure

```text
task6_project/
├── app/
│   └── main.py
├── models/
│   └── imdb_sentiment_api.keras
├── tests/
│   └── test_api.py
├── requirements.txt
├── Dockerfile
├── .dockerignore
├── .gitignore
└── README.md

Writing /content/task6_project/README.md


In [28]:
import os

project_dir = "/content/task6_project"

print("TASK 6 PROJECT FILE CHECK")
print("=" * 40)

required_files = [
    "app/main.py",
    "models/imdb_sentiment_api.keras",
    "tests/test_api.py",
    "requirements.txt",
    "Dockerfile",
    ".dockerignore",
    ".gitignore",
    "README.md"
]

all_found = True

for filename in required_files:
    filepath = os.path.join(project_dir, filename)

    if os.path.isfile(filepath):
        size = os.path.getsize(filepath) / (1024 * 1024)
        print(f"✅ {filename} ({size:.2f} MB)")
    else:
        print(f"❌ MISSING: {filename}")
        all_found = False

print("=" * 40)

if all_found:
    print("🎉 All required project files are present!")
else:
    print("⚠️ Some files are missing. We need to fix them.")

TASK 6 PROJECT FILE CHECK
✅ app/main.py (0.00 MB)
✅ models/imdb_sentiment_api.keras (15.04 MB)
✅ tests/test_api.py (0.00 MB)
✅ requirements.txt (0.00 MB)
✅ Dockerfile (0.00 MB)
✅ .dockerignore (0.00 MB)
✅ .gitignore (0.00 MB)
✅ README.md (0.00 MB)
🎉 All required project files are present!


In [33]:
import sys
import importlib.metadata as metadata

print("Python version:", sys.version)
print("\nChecking required packages:\n")

packages = [
    "fastapi",
    "uvicorn",
    "tensorflow",
    "pydantic",
    "pytest",
    "httpx",
    "numpy"
]

for package in packages:
    try:
        version = metadata.version(package)
        print(f"✅ {package}: {version}")
    except metadata.PackageNotFoundError:
        print(f"❌ {package}: NOT INSTALLED")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

Checking required packages:

✅ fastapi: 0.115.12
✅ uvicorn: 0.34.0
✅ tensorflow: 2.21.0
✅ pydantic: 2.10.6
✅ pytest: 8.3.5
✅ httpx: 0.28.1
✅ numpy: 2.1.3
